# Generate Dataset: Groq API → Informal/Formal Pairs

Loads the 91 seed pairs from `data/processed/seed_pairs.csv` (produced by `LoadDataset.ipynb`), then uses the Groq API (Llama 4 Scout) to generate enough additional pairs to reach **500 total**.

Output: `data/processed/informal_formal.csv` — 500 rows, columns: `informal`, `formal`

**Requires:** `GROQ_API_KEY` set below. Get one free at https://console.groq.com/

In [ ]:
# !pip install groq pandas truecase

In [ ]:
from groq import Groq
import pandas as pd
import json, time, os, re, nltk, truecase

nltk.download('punkt_tab', quiet=True)

GROQ_API_KEY = 'YOUR_GROQ_API_KEY_HERE' # REPLACED WITH PLACEHOLDER
TARGET       = 500
BATCH_SIZE   = 10   # pairs to request per API call

client = Groq(api_key=GROQ_API_KEY)

In [38]:
df_seed = pd.read_csv('data/processed/seed_pairs.csv')
n_needed = TARGET - len(df_seed)
print(f'Seed pairs: {len(df_seed)}')
print(f'Need to generate: {n_needed} more pairs')

Seed pairs: 91
Need to generate: 409 more pairs


In [39]:
PROMPT_TEMPLATE = """
Generate {n} pairs of informal and formal English sentences.
Cover a variety of everyday topics (food, travel, work, technology, school, etc.).

Rules:
- The informal sentence should sound like casual spoken or texted English.
- The formal sentence must convey the exact same meaning in a professional, polished tone.
- Keep sentences concise (under 30 words each).
- Do not repeat topics across pairs.

Respond with a JSON array only, no extra text. Format:
[
  {{"informal": "...", "formal": "..."}},
  ...
]
"""

def generate_batch(n):
    """Request n pairs from Llama 4 Scout via Groq, return list of dicts."""
    prompt = PROMPT_TEMPLATE.format(n=n)
    response = client.chat.completions.create(
        model='meta-llama/llama-4-scout-17b-16e-instruct',
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048
    )
    text = response.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if text.startswith('```'):
        text = text.split('\n', 1)[1].rsplit('```', 1)[0]
    return json.loads(text)

In [40]:
generated = []
n_batches = -(-n_needed // BATCH_SIZE)  # ceiling division

for i in range(n_batches):
    remaining = n_needed - len(generated)
    batch_n   = min(BATCH_SIZE, remaining)
    print(f'Batch {i+1}/{n_batches}: requesting {batch_n} pairs...', end=' ')
    try:
        batch = generate_batch(batch_n)
        generated.extend(batch)
        print(f'OK ({len(generated)}/{n_needed} generated)')
    except Exception as e:
        print(f'FAILED: {e}')
    time.sleep(2)  # 2s between calls to stay within free tier limits

df_generated = pd.DataFrame(generated)[['informal', 'formal']]
print(f'\nGenerated {len(df_generated)} pairs')

Batch 1/41: requesting 10 pairs... OK (10/409 generated)
Batch 2/41: requesting 10 pairs... OK (20/409 generated)
Batch 3/41: requesting 10 pairs... OK (30/409 generated)
Batch 4/41: requesting 10 pairs... OK (40/409 generated)
Batch 5/41: requesting 10 pairs... OK (50/409 generated)
Batch 6/41: requesting 10 pairs... OK (60/409 generated)
Batch 7/41: requesting 10 pairs... OK (70/409 generated)
Batch 8/41: requesting 10 pairs... OK (80/409 generated)
Batch 9/41: requesting 10 pairs... OK (90/409 generated)
Batch 10/41: requesting 10 pairs... OK (100/409 generated)
Batch 11/41: requesting 10 pairs... OK (110/409 generated)
Batch 12/41: requesting 10 pairs... OK (120/409 generated)
Batch 13/41: requesting 10 pairs... OK (130/409 generated)
Batch 14/41: requesting 10 pairs... OK (140/409 generated)
Batch 15/41: requesting 10 pairs... OK (150/409 generated)
Batch 16/41: requesting 10 pairs... OK (160/409 generated)
Batch 17/41: requesting 10 pairs... OK (170/409 generated)
Batch 18/41: re

In [41]:
df_all = pd.concat([df_seed, df_generated], ignore_index=True)
df_all.drop_duplicates(subset='informal', inplace=True)
df_all = df_all.head(TARGET).reset_index(drop=True)

def clean(text):
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    text = truecase.get_true_case(text)             # restore proper capitalization
    if text and text[-1] not in '.!?':
        text += '.'                                 # ensure closing punctuation
    return text

df_all['informal'] = df_all['informal'].apply(clean)
df_all['formal']   = df_all['formal'].apply(clean)

df_all.to_csv('data/processed/informal_formal.csv', index=False)
print(f'Saved {len(df_all)} pairs to data/processed/informal_formal.csv')
df_all.sample(5)

Saved 365 pairs to data/processed/informal_formal.csv


,informal,formal
14,"Tablets are better than textbooks, teaching ch...","Facilitating students' capacity to research, a..."
147,Need to finish this project by Friday.,I have a deadline to complete this project by ...
34,One could put a spacecraft near an asteroid fo...,Earthlings could fly a spacecraft in the vicin...
283,"I'm skipping school today, it's boring.","I have decided not to attend school today, as ..."
178,"The Museum was amazing, loved the art.","The Museum was impressive, and I greatly enjoy..."
